# A correção de 2026: MERGE no Delta Lake

## Task 4: corrigir os placares de 2026 sem recarregar a Copa

Nas tabelas do vídeo 5.3, 2026 aparece com **333 gols**; o oficial é **308**. Nas partidas decididas nos pênaltis, a fonte de 2026 soma os gols
da disputa no placar (`data/README.md`). O fornecedor mandou um **lote de correção** em `data/raw_2026_correcao`, com
as 4 partidas.

In [ ]:
import sys
sys.path.insert(0, "../../src")

from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession, functions as F

construtor = (
    SparkSession.builder
    .appName("arena-correcao")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)
spark = configure_spark_with_delta_pip(construtor).getOrCreate()

RAW = "../../data/raw"
RAW_CORRECAO = "../../data/raw_2026_correcao"
OUT = "../../data/out"

spark

A célula abaixo já vem pronta: `ler_partidas` (vídeo 5.1), `gravar` (vídeo 5.3) e as funções do resumo.

In [ ]:
# vídeo 5.1: a leitura das partidas
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

def ler_partidas(pasta):
    return limpar_partidas(
        spark.read.option("header", "true").schema(MATCHES).csv(f"{pasta}/WorldCupMatches.csv")
    )

# vídeo 5.3: a gravação das tabelas de verdade
def gravar(df, tabela):
    (
        df.repartition("ano").write.format("delta").mode("overwrite")
        .option("partitionOverwriteMode", "dynamic")
        .partitionBy("ano")
        .save(f"{OUT}/{tabela}")
    )

from wc_pipeline.transformacoes import partidas_por_selecao, resumo_por_selecao

## O placar de 2026 hoje

In [ ]:
partidas = spark.read.format("delta").load(f"{OUT}/partidas")
partidas.filter(F.col("ano") == 2026).select(F.sum("total_gols")).show()

Agora o lote de correção, com a mesma `ler_partidas`: ela recalcula `total_gols` e `resultado`.

In [ ]:
correcao = ler_partidas(RAW_CORRECAO)
correcao.select("match_id", "fase", "mandante", "gols_mandante", "gols_visitante", "visitante", "resultado").show()

## Por que não o overwrite?

O overwrite dinâmico troca a **pasta inteira** de 2026: ela ficaria só com as 4 partidas do lote. O lote é **parcial**:
é o caso do `MERGE`.

## MERGE INTO

| A chave do lote... | O que o MERGE faz | Como se escreve |
|---|---|---|
| já existe na tabela | atualiza a linha | `WHEN MATCHED THEN UPDATE SET *` |
| não existe na tabela | insere a linha | `WHEN NOT MATCHED THEN INSERT *` |

A chave é o `match_id`.

In [ ]:
from pathlib import Path

correcao.createOrReplaceTempView("correcao")
caminho = Path(f"{OUT}/partidas").resolve()

spark.sql(f"""
    MERGE INTO delta.`{caminho}` AS destino
    USING correcao AS lote
    ON destino.match_id = lote.match_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""").show()

4 linhas atualizadas, nenhuma inserida.

## Conferindo a correção

In [ ]:
partidas = spark.read.format("delta").load(f"{OUT}/partidas")
print(partidas.count(), "partidas")
partidas.filter(F.col("ano") == 2026).select(F.sum("total_gols")).show()

In [ ]:
(
    partidas.join(correcao.select("match_id"), on="match_id")
    .select("mandante", "gols_mandante", "gols_visitante", "visitante", "resultado")
    .show()
)

2026 com 308 gols, e as 4 partidas agora são **empate**, como no histórico.

## A correção no histórico da tabela

In [ ]:
spark.sql(f"DESCRIBE HISTORY delta.`{caminho}`").select("version", "operation").show()

In [ ]:
antes = spark.read.format("delta").option("versionAsOf", 1).load(f"{OUT}/partidas")
antes.filter(F.col("ano") == 2026).select(F.sum("total_gols")).show()

Três versões: o histórico, 2026 e a correção. A versão 1 continua com os 333 gols.

## O resumo por seleção

In [ ]:
resumo = spark.read.format("delta").load(f"{OUT}/resumo_selecoes")
resumo.filter((F.col("ano") == 2026) & F.col("selecao").isin("Germany", "Paraguay")).show()

O resumo é uma **agregação**: recalcule 2026 a partir das partidas corrigidas e grave só 2026, com o overwrite
dinâmico da função `gravar`.

In [ ]:
partidas_2026 = partidas.filter(F.col("ano") == 2026)
gravar(resumo_por_selecao(partidas_por_selecao(partidas_2026)), "resumo_selecoes")

resumo = spark.read.format("delta").load(f"{OUT}/resumo_selecoes")
resumo.filter((F.col("ano") == 2026) & F.col("selecao").isin("Germany", "Paraguay")).show()

Empate para as duas. **Linhas** de um lote parcial: `MERGE`. **Agregações**: recalcular e regravar a parte que mudou.

## Conferindo a saída

In [ ]:
gols_partidas = partidas.select(F.sum("total_gols")).first()[0]
gols_selecoes = resumo.select(F.sum("gols_pro")).first()[0]
gols_2026 = partidas_2026.select(F.sum("total_gols")).first()[0]

assert partidas.count() == 964 + 104, "o total de partidas mudou"
assert gols_2026 == 308, "2026 deveria ter os 308 gols oficiais"
assert gols_partidas == gols_selecoes, "os gols das partidas e das seleções não batem"

print(f"saída conferida: {partidas.count()} partidas, {gols_partidas} gols, {gols_2026} em 2026")

## A consulta da aula 3, agora com 2026 corrigida

In [ ]:
partidas.createOrReplaceTempView("partidas")
consulta = Path("../../sql/gols_por_copa.sql").read_text(encoding="utf-8")
spark.sql(consulta).filter(F.col("ano") >= 2014).show()

2026 com 308 gols. O público vazio vem do arquivo, como 2018 e 2022 (vídeo 2.2).

## Demanda da aula 5 atendida

| Etapa | Entrega |
|---|---|
| Task 1 | `overwrite` em Delta Lake |
| Task 2 | saída por Copa, com partition pruning |
| Task 3 | um arquivo por pasta e overwrite dinâmico |
| Task 4 | `MERGE INTO` nas partidas e o resumo de 2026 recalculado |

## Onde cada coisa foi parar

| Aula | Onde ficou | Como é usada a partir de agora |
|---|---|---|
| 1 | `src/wc_pipeline/schemas.py` | cada lote novo é lido com o mesmo schema |
| 2 | `src/wc_pipeline/transformacoes.py` | as cargas e as correções passam pelas mesmas funções |
| 3 | `sql/gols_por_copa.sql` | analytics roda a consulta sobre as partidas consolidadas |
| 4 | `data/out/passes_*`, em Parquet | estudo pontual, refeito só se pedirem |
| 5 | `data/out/partidas` e `data/out/resumo_selecoes`, em Delta | Copa nova: overwrite dinâmico; correção: `MERGE` |

Próximo passo: a atividade "Subindo o projeto no Git".

## Para consultar depois

- **Mais condições no `MERGE`:** `WHEN MATCHED AND <condição> THEN ...` atualiza só parte das linhas, `WHEN MATCHED THEN
  DELETE` apaga, e `WHEN NOT MATCHED BY SOURCE` trata as linhas da tabela que não vieram no lote (docs.delta.io, Table
  deletes, updates, and merges).
- **`MERGE` em Python:** `DeltaTable.forPath(spark, caminho).merge(...)` faz o mesmo sem SQL.
- **Arquivos das versões antigas:** ficam na pasta até um `VACUUM`, que apaga os que não fazem mais parte das versões
  recentes (vídeo 5.1).
- **A carga sem notebook:** na empresa, o código da carga vira um script `.py`, executado com `spark-submit` (com
  `--packages` para o Delta) e chamado por um agendador, como o Airflow, a cada lote. A atividade "Subindo o projeto no
  Git" traz esse passo como opcional.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()